In [3]:
#The following file paths are all absolute paths. You can replace them with relative paths at runtime, and the files are located in their respective folders.
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
import matplotlib.pyplot as plt
import random
import argparse
import time
from collections import OrderedDict
from copy import copy
# import Learn_Knonlinear as lka
import scipy
import scipy.linalg
from scipy.integrate import odeint
from tqdm import tqdm, trange
import sys
import os
# os.chdir(r'/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/')
os.chdir(r'D:/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/')
sys.path.append("utility_LSPN/")
sys.path.append("LSPN_compare/sizeNN_learnmodel_train")
sys.path.append("utility_LSPN/")
try:
    from LSPN_test import LSPN_Mamba
except:
    pass
from Utility import data_collecter
os.environ['KMP_DUPLICATE_LIB_OK'] = "TRUE"

In [4]:
Methods = ["KNonlinear","KNonlinearRNN","KoopmanU",\
            "KoopmanNonlinearA","KoopmanNonlinear",\
                "KNonlinearmamba"]
Method_names = ["KDNN","KRNN","DKUC(no SOC)",\
            "DKAC(no SOC)","DKN(no SOC)","KNonlinearmamba"]

In [3]:
def eval_err(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "DATA/LSPN_Hyperparameter_data/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method_index==1:
        import Learn_Knonlinear_RNN as lka
    elif method_index==0:
        import Learn_Knonlinear as lka
    elif method_index==2:
        import Learn_DKUC_withoutSOC as lka
    elif method_index==3:
        import Learn_DKAC_withoutSOC as lka
    elif method_index==4:
        import Learn_DKN_withoutSOC as lka
    elif method_index==5:
        import Learn_Knonlinear_hyperparameter as lka
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+env_name) and file.endswith(".pth"):
            model_path = file  
    Data_collect = data_collecter(env_name)
    udim = Data_collect.udim
    Nstates = Data_collect.Nstates
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=udim)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=udim+Nstates,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim) 
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=3, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda") 
    net.load_state_dict(state_dict)
    total_params = sum(p.numel() for p in net.parameters())
    print(f"{method} Total parameters: {total_params}")
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 50
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 10
    max_loss_all = np.zeros((times,steps))
    mean_loss_all = np.zeros((times,steps))
    min_loss_all = np.zeros((times,steps))
    Test_time = np.zeros((times))
    test_loss = []
    test_loss_log = []
    with torch.no_grad():
        for i in trange(times, desc="predicting", unit="times"):
            start_time = time.time()
            test_data = Data_collect.collect_koopman_data(Samples,steps)
            # np.save("DATA/LSPN_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i),test_data)
            max_loss,mean_loss,min_loss = lka.K_loss(test_data,net,udim)
            max_loss_all[i] = max_loss.reshape(-1)
            mean_loss_all[i] = mean_loss.reshape(-1)
            test_loss.append(mean_loss.reshape(-1))
            test_loss_log.append(np.log10(mean_loss_all[i]))
            min_loss_all[i] = min_loss.reshape(-1)
            end_time = time.time()
            t_cost = end_time - start_time
            Test_time[i] = t_cost
            print(Test_time[i])
    max_mean = np.mean(max_loss_all,axis=0)
    max_std = np.std(max_loss_all,axis=0)
    mean_mean =  np.mean(mean_loss_all,axis=0)
    mean_std =  np.std(mean_loss_all,axis=0)
    min_mean =  np.mean(min_loss_all,axis=0)
    min_std =  np.std(min_loss_all,axis=0)  
    test_loss = np.array(test_loss)
    test_loss_log = np.array(test_loss_log)
    print("test_loss:{}".format(test_loss_log.shape))
    print("test_time:{}".format(Test_time.shape))
    # np.save("DATA/LSPN_compare_drawdata/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([max_mean,max_std,mean_mean,mean_std,min_mean,min_std]))
    np.save("DATA/LSPN_Hyperparameter_data/log_"+suffix+"{}".format(method)+"_{}.npy".format(steps),test_loss_log)
    np.save("DATA/LSPN_Hyperparameter_data/"+suffix+"{}".format(method)+"_{}.npy".format(steps),test_loss)
    np.save("DATA/LSPN_Hyperparameter_data/time_"+suffix+"{}".format(method)+"_{}.npy".format(steps),Test_time)
    return max_mean,max_std,mean_mean,mean_std,min_mean,min_std

In [4]:
suffix = ["*","*","*","*","*","mamba_test6"]#based and best 50 100 8 4 4
suffix = ["*","*","*","*","*","mamba_test_kstep100_kbatchsize100_d_state8_d_conv4_expand4"] 
suffix = ["*","*","*","*","*","mamba_test_kstep20_kbatchsize100_d_state8_d_conv4_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize75_d_state8_d_conv4_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize50_d_state8_d_conv4_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state4_d_conv4_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state2_d_conv4_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv2_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv3_expand4"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv4_expand2"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv4_expand6"]
suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv4_expand4"]
env_name = "DampingPendulum"
steps = 200
for i in [5]:#0,1,2,3,4,5
    eval_err(suffix[i],env_name,method_index=i,layer_i=4, steps = steps)

KNonlinearmamba
KNonlinearmamba Total parameters: 504


predicting:  10%|█         | 1/10 [00:00<00:07,  1.28times/s]

0.7811503410339355


predicting:  20%|██        | 2/10 [00:01<00:05,  1.34times/s]

0.7211380004882812


...

0.7250034809112549


predicting:  90%|█████████ | 9/10 [00:06<00:00,  1.37times/s]

0.7319233417510986


predicting: 100%|██████████| 10/10 [00:07<00:00,  1.36times/s]

0.7266690731048584
test_loss:(10, 200)
test_time:(10,)


In [15]:
for k in [5]:
    suffix = ["*","*","*","*","*","mamba_test_kstep100_kbatchsize100_d_state8_d_conv4_expand4"] 
    suffix = ["*","*","*","*","*","mamba_test_kstep20_kbatchsize100_d_state8_d_conv4_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize75_d_state8_d_conv4_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize50_d_state8_d_conv4_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state4_d_conv4_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state2_d_conv4_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv2_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv3_expand4"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv4_expand2"]
    suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv4_expand6"]
    # suffix = ["*","*","*","*","*","mamba_test_kstep50_kbatchsize100_d_state8_d_conv4_expand4"]
    env_name = "DampingPendulum"
    method = Methods[k]
    steps = 200
    data_all = np.load("DATA/LSPN_Hyperparameter_data/"+suffix[k]+"{}".format(method)+"_{}.npy".format(steps))
    time_cost = np.load("DATA/LSPN_Hyperparameter_data/time_"+suffix[k]+"{}".format(method)+"_{}.npy".format(steps))
    Loss_Data = []
    mean_data = []
    for i in range(steps):#15
        data = data_all[:,i]
        # 计算平均值和标准差
        mean_value = sum(data) / len(data)
        std_deviation = (sum((x - mean_value) ** 2 for x in data) / len(data)) ** 0.5

        # 使用科学计数法格式化字符串
        formatted_string = "{:.3e} ± {:.3e}".format(mean_value, std_deviation)
        Loss_Data.append(formatted_string)
        # print(formatted_string)
        mean_data.append(mean_value)
    Loss_Data = np.array(Loss_Data)
    mean_data = np.array(mean_data)
    print(Loss_Data[9],Loss_Data[49],Loss_Data[99],Loss_Data[199])
    # np.save("DATA/LSPN_compare_drawdata/{}_{}step_loss.npy".format(method, steps),Loss_Data)

2.103e-03 ± 2.045e-04 7.774e-03 ± 1.411e-03 2.126e-02 ± 5.659e-03 4.685e-02 ± 9.830e-03
